# Smart Agriculture AI — Dataset Audit

Run this notebook on your **updated crop and weather datasets** to collect the exact statistics needed for the IEEE report.

It reports:

- total images
- class-wise counts and percentages
- ImageFolder class-to-index mapping
- train/validation/test split counts when split folders exist
- image dimensions and aspect ratios
- image modes (RGB/RGBA/grayscale)
- file extensions
- corrupted/unreadable images
- exact duplicate files
- class-imbalance statistics
- sample images for visual checking
- report-ready CSV, JSON, and TXT outputs

**Important:** the notebook reports the dataset that you actually run. It does not substitute the reference-paper dataset counts.


In [ ]:
# Install dependencies if you are using Google Colab.
# Run only once.

!pip -q install pillow pandas matplotlib numpy


In [ ]:
from pathlib import Path
import json
import hashlib
import random
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, UnidentifiedImageError

# ============================================================
# EDIT THESE TWO PATHS
# ============================================================

CROP_ROOT = Path("/datasets/crop")
WEATHER_ROOT = Path("/datasets/weather")

# Example for Google Drive:
# from google.colab import drive
# drive.mount('/content/drive')
# CROP_ROOT = Path('/content/drive/MyDrive/SmartAgriculture/datasets/crop')
# WEATHER_ROOT = Path('/content/drive/MyDrive/SmartAgriculture/datasets/weather')

OUTPUT_ROOT = Path("/smart_agriculture_dataset_audit")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

IMAGE_EXTENSIONS = {
    ".jpg", ".jpeg", ".png", ".bmp", ".gif", ".webp", ".tif", ".tiff"
}

SAMPLES_PER_CLASS = 5
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print("Crop root   :", CROP_ROOT)
print("Weather root:", WEATHER_ROOT)
print("Output root :", OUTPUT_ROOT)


In [ ]:
# Check that the paths are correct.

for name, root in [("CROP", CROP_ROOT), ("WEATHER", WEATHER_ROOT)]:
    print("=" * 70)
    print(name, "ROOT")
    print("Path   :", root)
    print("Exists :", root.exists())
    print("Folder :", root.is_dir() if root.exists() else False)

    if root.exists() and root.is_dir():
        print("Top-level items:")
        for p in sorted(root.iterdir())[:30]:
            print("  ", p.name)
    print()


In [ ]:
def is_image_file(path):
    return path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS


def iter_image_files(root):
    if not root.exists():
        return []
    return sorted([p for p in root.rglob("*") if is_image_file(p)])


def detect_split_structure(root):
    # Detect common split-folder names.
    aliases = {
        "train": {"train", "training"},
        "val": {"val", "valid", "validation"},
        "test": {"test", "testing"},
    }

    if not root.exists():
        return {}

    top_dirs = {p.name.lower(): p for p in root.iterdir() if p.is_dir()}
    detected = {}

    for canonical, names in aliases.items():
        for name in names:
            if name in top_dirs:
                detected[canonical] = top_dirs[name]
                break

    return detected


def get_split_for_file(file_path, root):
    split_dirs = detect_split_structure(root)

    if not split_dirs:
        return "unsplit"

    rel = file_path.relative_to(root)
    first = rel.parts[0].lower()

    if first in {"train", "training"}:
        return "train"
    if first in {"val", "valid", "validation"}:
        return "val"
    if first in {"test", "testing"}:
        return "test"

    return "unknown"


def get_class_for_file(file_path, root):
    rel = file_path.relative_to(root)
    parts = rel.parts
    split_dirs = detect_split_structure(root)

    if split_dirs and len(parts) >= 3:
        return parts[1]

    if len(parts) >= 2:
        return parts[0]

    return "UNKNOWN"


def md5_file(path, chunk_size=1024 * 1024):
    h = hashlib.md5()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()


def inspect_image(path):
    try:
        with Image.open(path) as img:
            img.verify()

        with Image.open(path) as img:
            return {
                "readable": True,
                "width": img.width,
                "height": img.height,
                "mode": img.mode,
                "format": img.format
            }

    except (UnidentifiedImageError, OSError, ValueError):
        return {
            "readable": False,
            "width": None,
            "height": None,
            "mode": None,
            "format": None
        }


def audit_dataset(root, dataset_name):
    files = iter_image_files(root)
    rows = []

    for path in files:
        info = inspect_image(path)
        rows.append({
            "dataset": dataset_name,
            "path": str(path),
            "relative_path": str(path.relative_to(root)),
            "filename": path.name,
            "extension": path.suffix.lower(),
            "split": get_split_for_file(path, root),
            "class": get_class_for_file(path, root),
            "readable": info["readable"],
            "width": info["width"],
            "height": info["height"],
            "mode": info["mode"],
            "format": info["format"],
            "aspect_ratio": (
                info["width"] / info["height"]
                if info["readable"] and info["height"] else np.nan
            ),
            "file_size_bytes": path.stat().st_size
        })

    df = pd.DataFrame(rows)

    if not df.empty:
        hashes = []
        for path, ok in zip(df["path"], df["readable"]):
            hashes.append(md5_file(Path(path)) if ok else None)
        df["md5"] = hashes
    else:
        df["md5"] = pd.Series(dtype="object")

    return df


def class_summary(df):
    if df.empty:
        return pd.DataFrame(columns=["class", "image_count", "percentage"])

    valid = df[df["readable"]].copy()
    total = len(valid)

    out = (
        valid.groupby("class")
        .size()
        .reset_index(name="image_count")
        .sort_values(["image_count", "class"], ascending=[False, True])
    )

    out["percentage"] = (100 * out["image_count"] / total).round(2) if total else 0
    out["percentage_of_largest"] = (
        out["image_count"] / out["image_count"].max()
    ).round(3)

    return out


def split_summary(df):
    if df.empty:
        return pd.DataFrame(columns=["split", "class", "image_count", "percentage_within_split"])

    valid = df[df["readable"]].copy()

    out = (
        valid.groupby(["split", "class"])
        .size()
        .reset_index(name="image_count")
        .sort_values(["split", "class"])
    )

    totals = out.groupby("split")["image_count"].transform("sum")
    out["percentage_within_split"] = (100 * out["image_count"] / totals).round(2)

    return out


def duplicate_rows(df):
    if df.empty or "md5" not in df:
        return pd.DataFrame()

    valid = df[df["readable"] & df["md5"].notna()].copy()
    counts = valid["md5"].value_counts()
    duplicate_hashes = counts[counts > 1].index

    return valid[valid["md5"].isin(duplicate_hashes)].sort_values("md5")


def dataset_stats(df):
    if df.empty:
        return {}

    valid = df[df["readable"]].copy()

    return {
        "total_files": int(len(df)),
        "readable_images": int(len(valid)),
        "corrupt_or_unreadable": int((~df["readable"]).sum()),
        "number_of_classes": int(valid["class"].nunique()),
        "classes": sorted(valid["class"].dropna().unique().tolist()),
        "splits": sorted(valid["split"].unique().tolist()),
        "number_of_unique_dimensions": int(valid[["width", "height"]].drop_duplicates().shape[0]),
        "mean_width": float(valid["width"].mean()),
        "median_width": float(valid["width"].median()),
        "min_width": int(valid["width"].min()),
        "max_width": int(valid["width"].max()),
        "mean_height": float(valid["height"].mean()),
        "median_height": float(valid["height"].median()),
        "min_height": int(valid["height"].min()),
        "max_height": int(valid["height"].max()),
        "mean_aspect_ratio": float(valid["aspect_ratio"].mean()),
        "image_modes": valid["mode"].value_counts().to_dict(),
        "extensions": valid["extension"].value_counts().to_dict(),
        "exact_duplicate_rows": int(len(duplicate_rows(df)))
    }


def imagefolder_mapping(df):
    if df.empty:
        return pd.DataFrame(columns=["class_index", "class_name"])

    classes = sorted(df.loc[df["readable"], "class"].dropna().unique().tolist())

    return pd.DataFrame({
        "class_index": list(range(len(classes))),
        "class_name": classes
    })


In [ ]:
# Run the full audit.

crop_df = audit_dataset(CROP_ROOT, "crop")
weather_df = audit_dataset(WEATHER_ROOT, "weather")

print("Crop image files found   :", len(crop_df))
print("Weather image files found:", len(weather_df))


In [ ]:
# Dataset-level statistics

crop_stats = dataset_stats(crop_df)
weather_stats = dataset_stats(weather_df)

print("CROP DATASET")
for k, v in crop_stats.items():
    print(f"{k}: {v}")

print("\nWEATHER DATASET")
for k, v in weather_stats.items():
    print(f"{k}: {v}")


In [ ]:
# Class-wise image counts

crop_classes = class_summary(crop_df)
weather_classes = class_summary(weather_df)

print("CROP CLASS DISTRIBUTION")
display(crop_classes)

print("\nWEATHER CLASS DISTRIBUTION")
display(weather_classes)

crop_classes.to_csv(OUTPUT_ROOT / "crop_class_distribution.csv", index=False)
weather_classes.to_csv(OUTPUT_ROOT / "weather_class_distribution.csv", index=False)


In [ ]:
# ImageFolder-style label mapping.
# torchvision ImageFolder uses alphabetical class ordering.

crop_mapping = imagefolder_mapping(crop_df)
weather_mapping = imagefolder_mapping(weather_df)

print("CROP CLASS MAPPING")
display(crop_mapping)

print("\nWEATHER CLASS MAPPING")
display(weather_mapping)

crop_mapping.to_csv(OUTPUT_ROOT / "crop_class_mapping.csv", index=False)
weather_mapping.to_csv(OUTPUT_ROOT / "weather_class_mapping.csv", index=False)


In [ ]:
# Train / validation / test distribution.
# This works automatically when your directory structure contains
# train, val/validation, and/or test folders.

crop_splits = split_summary(crop_df)
weather_splits = split_summary(weather_df)

print("CROP SPLITS")
display(crop_splits)

print("\nWEATHER SPLITS")
display(weather_splits)

crop_splits.to_csv(OUTPUT_ROOT / "crop_split_distribution.csv", index=False)
weather_splits.to_csv(OUTPUT_ROOT / "weather_split_distribution.csv", index=False)

def split_totals(df):
    if df.empty:
        return pd.DataFrame()

    valid = df[df["readable"]]
    out = valid.groupby("split").size().reset_index(name="images")
    total = out["images"].sum()

    if total:
        out["percentage"] = (100 * out["images"] / total).round(2)

    return out

crop_split_totals = split_totals(crop_df)
weather_split_totals = split_totals(weather_df)

print("\nCROP SPLIT TOTALS")
display(crop_split_totals)

print("\nWEATHER SPLIT TOTALS")
display(weather_split_totals)

crop_split_totals.to_csv(OUTPUT_ROOT / "crop_split_totals.csv", index=False)
weather_split_totals.to_csv(OUTPUT_ROOT / "weather_split_totals.csv", index=False)


In [ ]:
# Image dimensions, modes, and extensions

def audit_quality_tables(df, name):
    if df.empty:
        return

    valid = df[df["readable"]].copy()

    print(f"=== {name}: DIMENSIONS ===")
    dim = (
        valid.groupby(["width", "height"])
        .size()
        .reset_index(name="count")
        .sort_values("count", ascending=False)
    )
    display(dim.head(50))

    print(f"=== {name}: MODES ===")
    modes = valid["mode"].value_counts().rename_axis("mode").reset_index(name="count")
    display(modes)

    print(f"=== {name}: EXTENSIONS ===")
    ext = valid["extension"].value_counts().rename_axis("extension").reset_index(name="count")
    display(ext)

    print(f"=== {name}: FILE SIZE DESCRIPTION ===")
    display(valid["file_size_bytes"].describe().to_frame("bytes"))

    print(f"=== {name}: 128x128 CHECK ===")
    count_128 = ((valid["width"] == 128) & (valid["height"] == 128)).sum()
    total = len(valid)
    pct = 100 * count_128 / total if total else 0
    print(f"Already 128x128: {count_128}/{total} ({pct:.2f}%)")

    dim.to_csv(OUTPUT_ROOT / f"{name.lower()}_dimensions.csv", index=False)
    modes.to_csv(OUTPUT_ROOT / f"{name.lower()}_modes.csv", index=False)
    ext.to_csv(OUTPUT_ROOT / f"{name.lower()}_extensions.csv", index=False)

audit_quality_tables(crop_df, "CROP")
audit_quality_tables(weather_df, "WEATHER")


In [ ]:
# Corrupted / unreadable image files

def corrupted_report(df, name):
    if df.empty:
        return

    bad = df[~df["readable"]].copy()

    print(f"{name} corrupted/unreadable files: {len(bad)}")

    if not bad.empty:
        display(bad[["path", "relative_path", "class", "extension"]])
        bad.to_csv(OUTPUT_ROOT / f"{name.lower()}_corrupted_images.csv", index=False)

corrupted_report(crop_df, "CROP")
corrupted_report(weather_df, "WEATHER")


In [ ]:
# Exact duplicate analysis and cross-split leakage check

def duplicate_report(df, name):
    dup = duplicate_rows(df)

    print(f"=== {name}: DUPLICATE ANALYSIS ===")
    print("Duplicate file rows:", len(dup))

    if dup.empty:
        print("No exact duplicate image contents detected.")
        return

    display(dup[["split", "class", "relative_path", "md5"]].head(100))
    dup.to_csv(OUTPUT_ROOT / f"{name.lower()}_exact_duplicates.csv", index=False)

    cross_split = []
    for md5, group in dup.groupby("md5"):
        splits = sorted(group["split"].unique().tolist())
        if len(splits) > 1:
            cross_split.append({
                "md5": md5,
                "splits": ", ".join(splits),
                "classes": ", ".join(sorted(group["class"].unique())),
                "file_count": len(group)
            })

    cross_split_df = pd.DataFrame(cross_split)

    print("Cross-split duplicate groups:", len(cross_split_df))

    if not cross_split_df.empty:
        display(cross_split_df)
        cross_split_df.to_csv(
            OUTPUT_ROOT / f"{name.lower()}_cross_split_duplicates.csv",
            index=False
        )

duplicate_report(crop_df, "CROP")
duplicate_report(weather_df, "WEATHER")


In [ ]:
# Class balance report

def imbalance_report(summary_df, name):
    if summary_df.empty:
        return

    counts = summary_df["image_count"].to_numpy()

    min_count = counts.min()
    max_count = counts.max()
    mean_count = counts.mean()
    median_count = np.median(counts)
    imbalance_ratio = max_count / min_count if min_count else np.nan

    print(f"=== {name}: CLASS BALANCE ===")
    print(f"Number of classes       : {len(counts)}")
    print(f"Minimum class count     : {min_count}")
    print(f"Maximum class count     : {max_count}")
    print(f"Mean class count        : {mean_count:.2f}")
    print(f"Median class count      : {median_count:.2f}")
    print(f"Maximum / minimum ratio : {imbalance_ratio:.3f}")

imbalance_report(crop_classes, "CROP")
print()
imbalance_report(weather_classes, "WEATHER")


In [ ]:
# Visual inspection of random samples from every class

def show_samples(df, name, samples_per_class=5):
    if df.empty:
        print(f"{name}: no images found.")
        return

    valid = df[df["readable"]].copy()
    classes = sorted(valid["class"].unique())

    for cls in classes:
        paths = valid.loc[valid["class"] == cls, "path"].tolist()
        random.shuffle(paths)
        paths = paths[:samples_per_class]

        fig, axes = plt.subplots(1, len(paths), figsize=(3 * len(paths), 3))
        if len(paths) == 1:
            axes = [axes]

        fig.suptitle(f"{name} — {cls}")

        for ax, path in zip(axes, paths):
            try:
                with Image.open(path) as img:
                    ax.imshow(img.convert("RGB"))
                ax.set_title(Path(path).name[:18])
                ax.axis("off")
            except Exception:
                ax.set_title("Read error")
                ax.axis("off")

        plt.tight_layout()
        plt.show()

show_samples(crop_df, "CROP", SAMPLES_PER_CLASS)
show_samples(weather_df, "WEATHER", SAMPLES_PER_CLASS)


In [ ]:
# Class distribution plots

def plot_distribution(summary_df, title):
    if summary_df.empty:
        return

    plt.figure(figsize=(10, 5))
    plt.bar(summary_df["class"], summary_df["image_count"])
    plt.title(title)
    plt.xlabel("Class")
    plt.ylabel("Number of Images")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()

plot_distribution(crop_classes, "Crop Dataset — Class Distribution")
plot_distribution(weather_classes, "Weather Dataset — Class Distribution")


In [ ]:
# Create report-ready text summaries.

def make_report_text(name, df, summary_df):
    stats = dataset_stats(df)

    lines = [
        f"{name} DATASET — REPORT-READY SUMMARY",
        "=" * 70,
        f"Total image files discovered: {stats.get('total_files')}",
        f"Readable images: {stats.get('readable_images')}",
        f"Corrupted/unreadable images: {stats.get('corrupt_or_unreadable')}",
        f"Number of classes: {stats.get('number_of_classes')}",
        f"Classes: {', '.join(stats.get('classes', []))}",
        f"Detected splits: {', '.join(stats.get('splits', []))}",
        f"Unique image dimensions: {stats.get('number_of_unique_dimensions')}",
        f"Mean width: {stats.get('mean_width')}",
        f"Mean height: {stats.get('mean_height')}",
        f"Minimum width: {stats.get('min_width')}",
        f"Maximum width: {stats.get('max_width')}",
        f"Minimum height: {stats.get('min_height')}",
        f"Maximum height: {stats.get('max_height')}",
        f"Mean aspect ratio: {stats.get('mean_aspect_ratio')}",
        f"Image modes: {stats.get('image_modes')}",
        f"Exact duplicate rows: {stats.get('exact_duplicate_rows')}",
        "",
        "CLASS DISTRIBUTION",
        "-" * 70
    ]

    for _, row in summary_df.iterrows():
        lines.append(
            f"{row['class']}: {int(row['image_count'])} images "
            f"({row['percentage']:.2f}%)"
        )

    return "\n".join(lines)

crop_text = make_report_text("CROP", crop_df, crop_classes)
weather_text = make_report_text("WEATHER", weather_df, weather_classes)

print(crop_text)
print("\n")
print(weather_text)

(OUTPUT_ROOT / "crop_report_summary.txt").write_text(crop_text, encoding="utf-8")
(OUTPUT_ROOT / "weather_report_summary.txt").write_text(weather_text, encoding="utf-8")


In [ ]:
# Save complete machine-readable audit outputs.

crop_df.to_csv(OUTPUT_ROOT / "crop_image_level_audit.csv", index=False)
weather_df.to_csv(OUTPUT_ROOT / "weather_image_level_audit.csv", index=False)

(OUTPUT_ROOT / "crop_dataset_stats.json").write_text(
    json.dumps(crop_stats, indent=2, default=str),
    encoding="utf-8"
)

(OUTPUT_ROOT / "weather_dataset_stats.json").write_text(
    json.dumps(weather_stats, indent=2, default=str),
    encoding="utf-8"
)

combined = {
    "crop": crop_stats,
    "weather": weather_stats
}

(OUTPUT_ROOT / "dataset_audit_summary.json").write_text(
    json.dumps(combined, indent=2, default=str),
    encoding="utf-8"
)

print("Saved audit files:")
for p in sorted(OUTPUT_ROOT.iterdir()):
    print(" -", p.name)


In [ ]:
# OPTIONAL: verify torchvision ImageFolder class ordering.
# This is useful before writing the class-index mapping into the report.

try:
    from torchvision.datasets import ImageFolder

    if CROP_ROOT.exists() and not detect_split_structure(CROP_ROOT):
        ds = ImageFolder(str(CROP_ROOT))
        print("CROP ImageFolder classes :", ds.classes)
        print("CROP class_to_idx        :", ds.class_to_idx)

    if WEATHER_ROOT.exists() and not detect_split_structure(WEATHER_ROOT):
        ds = ImageFolder(str(WEATHER_ROOT))
        print("WEATHER ImageFolder classes :", ds.classes)
        print("WEATHER class_to_idx        :", ds.class_to_idx)

except Exception as e:
    print("Torchvision check skipped:", e)


## What you should give me after running the notebook

Send me the output files, especially:

- `dataset_audit_summary.json`
- `crop_class_distribution.csv`
- `weather_class_distribution.csv`
- `crop_split_totals.csv`
- `weather_split_totals.csv`
- `crop_split_distribution.csv`
- `weather_split_distribution.csv`
- `crop_report_summary.txt`
- `weather_report_summary.txt`

Then the IEEE paper can contain the **exact current dataset size, class distribution, split ratios, image statistics, data-quality information, and duplicate checks** rather than placeholders.

Also send the actual training/validation/test results separately if you have them; dataset auditing alone cannot produce model accuracy, F1-score, or confusion matrices.
